# Quantitative Finance — Deep Formula Analysis

This notebook is a **companion guide** to `quant_finance_formulas.ipynb`.  
Here we go deeper: every symbol is defined, every formula is broken down step-by-step,
and the intuition behind *why* the math works is explained in plain language.

**No code in this notebook** — this is pure theory and understanding.

---
# 1. VOLATILITY
---

## 1.1 Realized Volatility

### The Formula

$$\sigma = \sqrt{\frac{1}{n-1} \sum_{i=1}^{n} (r_i - \bar{r})^2}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Units / Range |
|--------|---------|---------------|
| $\sigma$ (sigma) | **Realized volatility** — the final answer. It measures how spread out returns are. | Decimal (e.g. 0.20 = 20%). Always ≥ 0. |
| $n$ | **Number of observations** — how many return data points you have (e.g. 252 daily returns = 1 year). | Integer ≥ 2 |
| $r_i$ | **Individual return** for period $i$. Typically a log-return or simple return. | Decimal (e.g. 0.01 = +1%) |
| $\bar{r}$ | **Mean (average) return** — $\bar{r} = \frac{1}{n}\sum r_i$. The center of the distribution. | Same units as $r_i$ |
| $(r_i - \bar{r})$ | **Deviation** — how far return $i$ is from the average. Can be positive or negative. | Same units as $r_i$ |
| $(r_i - \bar{r})^2$ | **Squared deviation** — squaring removes the sign so positive and negative deviations both count. | |
| $\frac{1}{n-1}$ | **Bessel's correction** — dividing by $n-1$ instead of $n$ gives an unbiased estimate of the population variance from a sample. | |
| $\sqrt{\cdot}$ | **Square root** — brings squared units back to the original return units. | |

### How the Formula Works Step-by-Step

1. **Collect returns**: Gather your $n$ return values: $r_1, r_2, \ldots, r_n$
2. **Find the average**: Compute $\bar{r} = \frac{1}{n}\sum r_i$
3. **Measure each deviation**: For each $r_i$, compute how far it is from $\bar{r}$
4. **Square them**: $(r_i - \bar{r})^2$ — this makes all deviations positive and penalizes large deviations more
5. **Average the squared deviations**: Divide by $n-1$ to get the **variance**
6. **Take the square root**: This converts variance back into the same scale as the returns

### Intuition

Imagine daily returns as dots scattered around their average. Volatility is measuring **how far apart the dots are spread**. If all dots are clustered tightly → low volatility (calm market). If dots are all over the place → high volatility (turbulent market).

### Annualization

Daily volatility is small (often 1-2%). To annualize it:

$$\sigma_{\text{annual}} = \sigma_{\text{daily}} \times \sqrt{252}$$

Why $\sqrt{252}$? Because there are ~252 trading days per year, and if daily returns are independent, variance scales linearly with time, so standard deviation scales with the square root.

### Common Pitfalls

- Using $n$ instead of $n-1$ (biased estimate)
- Mixing daily vol with annual vol (always check the time scale)
- Assuming returns are normally distributed (fat tails exist in real markets)

## 1.2 Implied Volatility

### The Formula

$$\text{Solve: } C_{\text{market}} = C_{\text{BS}}(S, K, T, r, \sigma_{\text{impl}}) \quad \text{for } \sigma_{\text{impl}}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Units / Range |
|--------|---------|---------------|
| $C_{\text{market}}$ | **Market price of the option** — the actual price you see being traded. | Dollars (e.g. $5.50) |
| $C_{\text{BS}}$ | **Black-Scholes theoretical price** — the price the model predicts given the inputs. | Dollars |
| $S$ | **Current stock (spot) price** — what the underlying asset trades at right now. | Dollars |
| $K$ | **Strike price** — the price at which the option holder can buy (call) or sell (put) the asset. | Dollars |
| $T$ | **Time to expiration** — how long until the option expires. | Years (e.g. 0.5 = 6 months) |
| $r$ | **Risk-free interest rate** — the return on a "riskless" investment like a Treasury bill. | Decimal per year |
| $\sigma_{\text{impl}}$ | **Implied volatility** — the unknown we are solving for. It is the volatility that, when plugged into Black-Scholes, makes the theoretical price match the market price. | Decimal per year |

### How It Works

This is **not** a direct formula — it's an **inverse problem**:

1. You observe the market price of a call option ($C_{\text{market}}$)
2. You know $S$, $K$, $T$, and $r$ from the market
3. The only unknown in the Black-Scholes formula is $\sigma$
4. You ask: "What value of $\sigma$ makes $C_{\text{BS}} = C_{\text{market}}$?"
5. Since there's no closed-form solution, you use **numerical root-finding** (Brent's method)

### Why Brent's Method?

Brent's method is a root-finding algorithm that combines bisection (reliable but slow) with interpolation (fast but can fail). It finds the $\sigma$ where:

$$f(\sigma) = C_{\text{BS}}(S, K, T, r, \sigma) - C_{\text{market}} = 0$$

It works because $C_{\text{BS}}$ is a **monotonically increasing** function of $\sigma$ — higher volatility always means a higher option price. So there's always exactly one solution.

### Intuition

Think of implied volatility as the market's **collective opinion** about how much the stock will move. If options are expensive → implied vol is high → the market expects big moves (or is uncertain). If options are cheap → implied vol is low → calm expected.

### Key Insight: Forward-Looking vs Backward-Looking

- **Realized volatility** looks at the past: "How much *did* the stock move?"
- **Implied volatility** looks at the future: "How much *will* it move, according to the market?"

The gap between the two is called the **volatility risk premium**.

## 1.3 Forward Volatility

### The Formula

$$\sigma^2(T_1, T_2) = \frac{\sigma^2(0, T_2) \cdot T_2 - \sigma^2(0, T_1) \cdot T_1}{T_2 - T_1}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $\sigma^2(T_1, T_2)$ | **Forward variance** for the period from $T_1$ to $T_2$. Take the square root to get forward volatility. |
| $\sigma(0, T_1)$ | **Implied volatility** from now (time 0) to time $T_1$ — e.g. the 1-year implied vol. |
| $\sigma(0, T_2)$ | **Implied volatility** from now to time $T_2$ — e.g. the 2-year implied vol. |
| $T_1$ | **Start of the forward period** in years. |
| $T_2$ | **End of the forward period** in years. Must be > $T_1$. |
| $\sigma^2 \cdot T$ | **Total variance** — this is the key insight. Variance (not volatility) is additive over time. |

### How It Works

Think of variance like distance traveled:

1. **Total variance** from 0 to $T_2$ is $\sigma^2(0,T_2) \cdot T_2$
2. **Total variance** from 0 to $T_1$ is $\sigma^2(0,T_1) \cdot T_1$
3. **Subtract** to get the variance that belongs to the $[T_1, T_2]$ interval
4. **Divide** by $(T_2 - T_1)$ to get variance **per unit time** in that interval

$$\text{Forward Volatility} = \sqrt{\sigma^2(T_1, T_2)}$$

### Analogy

If you drive 100 km in 2 hours and 30 km in the first hour, your speed in the second hour was:
$(100 - 30) / (2 - 1) = 70$ km/h. Same logic — just with variance instead of distance.

### Why It Matters

Forward volatility is essential for pricing **forward-starting options** and for understanding what the market expects about future uncertainty across different time horizons.

## 1.4 Cumulative Return

### The Formula

$$R = \prod_{i=1}^{n}(1 + r_i) - 1$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $R$ | **Cumulative (total) return** over all $n$ periods. |
| $\prod$ | **Product operator** — multiply all the terms together (like $\sum$ is for addition). |
| $r_i$ | **Return in period $i$** (e.g. daily return). |
| $(1 + r_i)$ | **Growth factor** for period $i$. If $r_i = 0.03$ (+3%), the growth factor is 1.03. |
| $-1$ | Subtracting 1 converts the growth factor back to a return. |

### How It Works

1. Start with $1 (or 100%)
2. Period 1: multiply by $(1 + r_1)$ → your new balance
3. Period 2: multiply that result by $(1 + r_2)$ → new balance
4. Continue for all periods
5. Subtract 1 to get the net gain/loss as a percentage

### Why Multiply, Not Add?

Returns **compound**. If you gain 10% then lose 10%, you DON'T end up at zero. You end up at:
$(1.10)(0.90) - 1 = -0.01 = -1\%$. The loss is applied to a larger base (after the gain).

### Example Walkthrough

Three monthly returns: +5%, -3%, +2%

$$R = (1.05)(0.97)(1.02) - 1 = 1.03857 - 1 = 0.03857 = +3.857\%$$

Note: simple addition would give $5 - 3 + 2 = 4\%$, which overestimates the true compounded return.

---
# 2. RISK METRICS
---

## 2.1 Value at Risk (VaR) — Parametric

### The Formula

$$\text{VaR}_{\alpha} = \mu - z_{\alpha} \cdot \sigma$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Typical Values |
|--------|---------|----------------|
| $\text{VaR}_{\alpha}$ | **Value at Risk** at confidence level $\alpha$. The threshold loss that is exceeded only $(1-\alpha)$% of the time. | Expressed as a return (e.g. -0.032 = -3.2%) |
| $\alpha$ | **Confidence level** — the probability that losses will NOT exceed VaR. | Typically 0.95 (95%) or 0.99 (99%) |
| $\mu$ | **Expected (mean) return** of the portfolio over the period. | Daily: ~0.0004 (≈0.04%) |
| $z_{\alpha}$ | **Critical value** of the standard normal distribution at level $\alpha$. This is how many standard deviations from the mean defines the boundary. | $z_{0.95} = 1.645$, $z_{0.99} = 2.326$ |
| $\sigma$ | **Standard deviation** of portfolio returns. | Daily: ~0.01-0.03 |

### How It Works

The parametric VaR assumes returns follow a **normal distribution** $N(\mu, \sigma^2)$.

1. The normal distribution is symmetric and bell-shaped
2. $z_{\alpha}$ tells you how many sigmas you need to go left from the mean to capture the worst $\alpha$% of outcomes
3. $\mu - z_{\alpha} \cdot \sigma$ gives you that threshold return
4. Any loss beyond this point happens with probability $(1-\alpha)$

### Reading the Result

If $\text{VaR}_{0.95} = -0.032$ and your portfolio is $1M:

- There is a **95% chance** that daily losses will not exceed $\$32{,}000$
- Equivalently, there is a **5% chance** of losing **more** than $\$32{,}000$ in a day

### Limitations

- Assumes normality — real returns have **fat tails** (extreme events happen more often than normal distribution predicts)
- VaR doesn't say HOW BAD losses can get beyond the threshold (use **Expected Shortfall / CVaR** for that)
- It's a single number — can't capture the full risk profile

## 2.2 Sharpe Ratio

### The Formula

$$\text{Sharpe} = \frac{E[R] - R_f}{\sigma}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Typical Values |
|--------|---------|----------------|
| $\text{Sharpe}$ | **Sharpe ratio** — reward per unit of total risk. | < 0 = bad, 0.5-1 = decent, > 1 = good, > 2 = excellent |
| $E[R]$ | **Expected return** of the portfolio (annualized). | 0.08 to 0.15 (8%-15%) |
| $R_f$ | **Risk-free rate** — what you'd earn with zero risk (e.g. T-bill yield). | 0.02 to 0.05 |
| $E[R] - R_f$ | **Excess return** — the premium you earn for taking risk. This is the "reward" part. | |
| $\sigma$ | **Total volatility** (standard deviation) of portfolio returns. This is the "risk" part. | 0.10 to 0.25 |

### How It Works

The Sharpe ratio answers: **"For every unit of risk I take, how many units of return do I earn above the risk-free rate?"**

- Numerator = how much EXTRA return you get vs doing nothing (risk-free)
- Denominator = how much RISK you endure to get that extra return
- Higher ratio = better risk-adjusted performance

### Interpretation Table

| Sharpe | Interpretation |
|--------|----------------|
| < 0 | Worse than risk-free — losing money on a risk-adjusted basis |
| 0 - 0.5 | Poor — not enough reward for the risk |
| 0.5 - 1.0 | Acceptable — in line with many equity benchmarks |
| 1.0 - 2.0 | Good — strong risk-adjusted performance |
| > 2.0 | Excellent — typical of top-tier hedge funds (annualized) |

### Key Caveat

The Sharpe ratio treats **all** volatility as bad — including upside volatility. If your portfolio shoots up 30% in a month, the Sharpe ratio *penalizes* you for it. The Sortino ratio fixes this.

## 2.3 Sortino Ratio

### The Formula

$$\text{Sortino} = \frac{E[R] - R_f}{\sigma_{\text{down}}}$$

where

$$\sigma_{\text{down}} = \sqrt{\frac{1}{n_{\text{neg}}-1} \sum_{r_i < 0} r_i^2}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $\text{Sortino}$ | **Sortino ratio** — reward per unit of *downside* risk only. |
| $E[R]$ | **Expected return** of the portfolio. |
| $R_f$ | **Risk-free rate**. |
| $\sigma_{\text{down}}$ | **Downside deviation** — standard deviation computed using only negative returns. |
| $n_{\text{neg}}$ | Number of negative return observations. |

### Sharpe vs Sortino

| Aspect | Sharpe | Sortino |
|--------|--------|---------|
| Risk measure | Total volatility (up + down) | Only downside volatility |
| Penalizes upside? | Yes | No |
| Better for | Symmetric return distributions | Asymmetric/skewed returns |

### Intuition

Investors don't mind upside surprises — they mind losses. The Sortino ratio only counts the bad surprises as risk. If two portfolios have the same Sharpe but one has most of its volatility on the upside, the Sortino ratio will correctly rank it higher.

## 2.4 RAROC (Risk-Adjusted Return on Capital)

### The Formula

$$\text{RAROC} = \frac{\text{Expected Return}}{\text{Economic Capital}}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| RAROC | **Risk-Adjusted Return on Capital** — profitability per unit of risk capital. |
| Expected Return | The profit the business unit or trade is expected to generate (after costs). |
| Economic Capital | The capital reserved to absorb potential unexpected losses (essentially a risk buffer). |

### How Banks Use It

A bank has many business units (mortgages, trading, credit cards). Each generates different returns and requires different amounts of capital reserve. RAROC lets the bank compare them fairly:

- **Trading desk A**: earns $10M but needs $100M capital → RAROC = 10%
- **Trading desk B**: earns $5M but needs $20M capital → RAROC = 25%

Desk B is more capital-efficient, even though it earns less in absolute terms.

### Decision Rule

If $\text{RAROC} > \text{hurdle rate}$ (the company's minimum acceptable return), the activity creates value. If below, the capital should be reallocated elsewhere.

---
# 3. OPTIONS & GREEKS
---

## 3.1 Black-Scholes Call Price

### The Formula

$$C = S \cdot N(d_1) - K \cdot e^{-rT} \cdot N(d_2)$$

where

$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2) \cdot T}{\sigma \sqrt{T}}$$

$$d_2 = d_1 - \sigma \sqrt{T}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Units |
|--------|---------|-------|
| $C$ | **Call option price** — the fair premium to pay. | Dollars |
| $S$ | **Spot price** — current price of the underlying stock. | Dollars |
| $K$ | **Strike price** — the price at which the holder can buy the stock at expiry. | Dollars |
| $T$ | **Time to expiration**. | Years |
| $r$ | **Continuously compounded risk-free rate**. | Per year |
| $\sigma$ | **Volatility** of the stock's log-returns. | Per year |
| $N(x)$ | **Standard normal CDF** — the probability that a standard normal variable is ≤ $x$. Returns a value between 0 and 1. | Probability |
| $\ln(S/K)$ | **Log-moneyness** — how far in/out of the money the option is. Positive = in the money. | Dimensionless |
| $e^{-rT}$ | **Discount factor** — converts a future dollar amount to its present value. If $r=5\%$ and $T=1$, this is ~0.951. | Dimensionless |
| $d_1$ | A standardized measure combining moneyness, drift, and time. Roughly: how many standard deviations the stock is above the strike, adjusted for growth. | Dimensionless |
| $d_2$ | Same as $d_1$ but adjusted down by $\sigma\sqrt{T}$. | Dimensionless |

### Dissecting the Formula

The call price has **two terms**:

$$C = \underbrace{S \cdot N(d_1)}_{\text{what you expect to receive}} - \underbrace{K \cdot e^{-rT} \cdot N(d_2)}_{\text{what you expect to pay}}$$

- **Term 1** ($S \cdot N(d_1)$): The expected value of the stock you'll receive if the option ends in the money. $N(d_1)$ is the probability-weighted amount, adjusted for the stock's drift.

- **Term 2** ($K \cdot e^{-rT} \cdot N(d_2)$): The present value of the strike price you'll pay, weighted by the probability $N(d_2)$ that you'll actually exercise (i.e., that $S_T > K$).

### Understanding $d_1$ and $d_2$

$d_1$ numerator has three parts:

$$\underbrace{\ln(S/K)}_{\text{moneyness}} + \underbrace{r \cdot T}_{\text{risk-free growth}} + \underbrace{\frac{\sigma^2}{2} \cdot T}_{\text{volatility boost}}$$

The denominator $\sigma\sqrt{T}$ normalizes everything into "standard deviation units".

- $N(d_2) \approx$ probability the option expires in-the-money (under the risk-neutral measure)
- $N(d_1) > N(d_2)$ always, because $d_1 > d_2$

### Key Assumptions

1. The stock price follows geometric Brownian motion (log-normal distribution)
2. No dividends
3. Constant volatility and interest rate
4. No transaction costs
5. European exercise only (can only exercise at expiry)
6. Continuous trading is possible

## 3.2 Delta ($\Delta$)

### The Formula

$$\Delta = \frac{\partial C}{\partial S} = N(d_1)$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $\Delta$ (Delta) | **Rate of change** of the option price with respect to the stock price. |
| $\partial C / \partial S$ | **Partial derivative** of the call price $C$ with respect to $S$, holding everything else constant. |
| $N(d_1)$ | Standard normal CDF evaluated at $d_1$. |

### Properties

| Condition | Delta Range | Meaning |
|-----------|-------------|----------|
| Deep in-the-money ($S \gg K$) | $\Delta \to 1$ | Option moves $1 for every $1 stock move (behaves like stock) |
| At-the-money ($S \approx K$) | $\Delta \approx 0.5$ | Option moves ~$0.50 for every $1 stock move |
| Deep out-of-the-money ($S \ll K$) | $\Delta \to 0$ | Option barely reacts to stock moves |

### Three Interpretations of Delta

1. **Hedge ratio**: To delta-hedge a short call position, buy $\Delta$ shares per option sold
2. **Price sensitivity**: The option price changes by ~$\Delta$ dollars for a $1 move in the stock
3. **Probability proxy**: $\Delta \approx$ probability the option expires in-the-money (rough approximation)

## 3.3 Gamma ($\Gamma$)

### The Formula

$$\Gamma = \frac{\partial^2 C}{\partial S^2} = \frac{N'(d_1)}{S \cdot \sigma \cdot \sqrt{T}}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $\Gamma$ (Gamma) | **Rate of change of delta** as the stock price moves. The "acceleration" of the option price. |
| $\partial^2 C / \partial S^2$ | **Second partial derivative** of $C$ with respect to $S$. |
| $N'(d_1)$ | **Standard normal PDF** at $d_1$: $N'(x) = \frac{1}{\sqrt{2\pi}} e^{-x^2/2}$. This is the bell curve height. |
| $S \cdot \sigma \cdot \sqrt{T}$ | Normalizing factor — scales gamma by the stock's dollar volatility over the remaining life. |

### Intuition

- **Delta** tells you the slope (first derivative) — how the option moves with the stock
- **Gamma** tells you the curvature (second derivative) — how the slope itself changes

High gamma means delta is changing rapidly, which makes hedging difficult and expensive. Gamma is highest when:
- The option is **at-the-money** (most uncertainty about exercise)
- Time to expiry is **short** (delta must snap to 0 or 1 very quickly)

### Gamma Risk

If you are short gamma (sold options), large stock moves hurt you disproportionately because your hedge (delta) gets more wrong the bigger the move. This is why option sellers fear "gamma explosions" near expiry.

## 3.4 Vega ($\nu$)

### The Formula

$$\nu = \frac{\partial C}{\partial \sigma} = S \cdot N'(d_1) \cdot \sqrt{T}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $\nu$ (Vega) | **Sensitivity of option price to changes in volatility**. Not actually a Greek letter — it's from the Greek alphabet's neighbor! |
| $\partial C / \partial \sigma$ | Partial derivative of the call price with respect to volatility. |
| $S$ | Spot price — vega increases with higher stock price (more dollars at stake). |
| $N'(d_1)$ | Standard normal PDF at $d_1$ — highest when ATM. |
| $\sqrt{T}$ | Square root of time — more time = more sensitivity to vol (more time for vol to matter). |

### Convention

Vega is often quoted per 1% change in vol:

$$\text{If vega} = 40, \text{ a 1% vol increase } (20\% \to 21\%) \text{ raises the option by } \$0.40$$

### When Is Vega Highest?

- **ATM options**: Most sensitivity to vol changes
- **Long-dated options**: More time for volatility to play out
- **Higher stock prices**: More dollar value at stake

### Vega and Trading

- **Long vega**: You profit when volatility rises (e.g., long straddle before earnings)
- **Short vega**: You profit when volatility falls (e.g., selling options after a vol spike)

---
# 4. CREDIT RISK & EXECUTION
---

## 4.1 Loan-to-Value (LTV)

### The Formula

$$\text{LTV} = \frac{\text{Loan Amount}}{\text{Collateral Value}}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Example |
|--------|---------|----------|
| LTV | **Loan-to-Value ratio** — what fraction of the collateral is borrowed against. | 0.80 = 80% |
| Loan Amount | The total debt outstanding — what the borrower owes. | $320,000 |
| Collateral Value | The current market value of the asset securing the loan. | $400,000 |

### Interpretation

| LTV | Risk Level | Meaning |
|-----|------------|----------|
| < 50% | Low | Large equity cushion — collateral can drop 50% before loan is underwater |
| 50-80% | Moderate | Standard mortgage territory |
| 80-100% | High | Very little cushion — small price drop puts loan underwater |
| > 100% | Underwater | Borrower owes more than the collateral is worth |

### Equity Cushion

$$\text{Equity Cushion} = 1 - \text{LTV}$$

An LTV of 80% means a 20% equity cushion — the collateral must drop 20% before the loan exceeds the collateral value.

## 4.2 Expected Loss (EL)

### The Formula

$$\text{EL} = \text{PD} \times \text{LGD} \times \text{EAD}$$

### Symbol-by-Symbol Breakdown

| Symbol | Full Name | Meaning | Typical Range |
|--------|-----------|---------|---------------|
| EL | **Expected Loss** | The average loss over many scenarios. This is what you expect to lose "on average". | Dollars |
| PD | **Probability of Default** | The chance that the borrower fails to pay. Estimated from credit ratings, financial ratios, or models. | 0.001 (AAA) to 0.20 (junk) |
| LGD | **Loss Given Default** | If the borrower defaults, what fraction of the exposure do you lose? Accounts for recovery (selling collateral, etc.). | 0.20 to 0.60 typically |
| EAD | **Exposure at Default** | How much money is at risk when default happens. For a loan, it's the outstanding balance. For a credit line, it includes potential future drawdowns. | Dollars |

### The Logic Chain

$$\text{EL} = \underbrace{\text{PD}}_{\text{How likely is default?}} \times \underbrace{\text{LGD}}_{\text{How bad if default?}} \times \underbrace{\text{EAD}}_{\text{How much is at risk?}}$$

### Example

- A $1M loan to a BB-rated company: PD = 2%, LGD = 45%, EAD = $1M
- EL = 0.02 × 0.45 × $1M = $9,000
- The bank should provision at least $9,000 against this loan

### Important Note

EL is the **average** loss. Actual losses can be much higher (or zero). The variability around EL is called **Unexpected Loss (UL)**, and that's what Economic Capital is designed to cover.

## 4.3 TWAP (Time-Weighted Average Price)

### The Formula

$$\text{TWAP} = \frac{1}{T} \sum_{t=1}^{T} P(t)$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| TWAP | **Time-Weighted Average Price** — the simple arithmetic mean of prices over equal time intervals. |
| $T$ | **Number of time intervals** (e.g. 12 five-minute intervals in an hour). |
| $P(t)$ | **Price** at time interval $t$. |
| $1/T$ | Each time interval gets **equal weight**, regardless of trading volume. |

### How Traders Use TWAP

A TWAP algorithm splits a large order into equal-sized slices and executes one slice per time interval:

- Want to buy 12,000 shares over 1 hour?
- Split into 12 × 1,000-share orders, one every 5 minutes
- The average fill price should approximate the TWAP

### TWAP vs VWAP

| | TWAP | VWAP |
|---|------|------|
| Weights | Equal (time-based) | Volume-based |
| Use case | When you want to spread orders evenly over time | When you want to match the market's trading pattern |
| Manipulation | Harder to game (predictable schedule) | Can be gamed by trading around known VWAP targets |

## 4.4 VWAP (Volume-Weighted Average Price)

### The Formula

$$\text{VWAP} = \frac{\sum_{i=1}^{n} P_i \times V_i}{\sum_{i=1}^{n} V_i}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| VWAP | **Volume-Weighted Average Price** — the average price weighted by how much was traded at each price level. |
| $P_i$ | **Price** at interval $i$. |
| $V_i$ | **Volume** traded at interval $i$ (number of shares). |
| $P_i \times V_i$ | **Dollar volume** at interval $i$ — total money transacted. |
| $\sum V_i$ | **Total volume** — sum of all shares traded. |

### Why Volume Matters

If a stock trades at $100 for 1,000 shares and then at $110 for 9,000 shares:

- TWAP = $(100 + 110) / 2 = \$105$
- VWAP = $(100 \times 1000 + 110 \times 9000) / (1000 + 9000) = \$109$

VWAP more accurately reflects the price that *most participants actually paid*.

### Benchmark Usage

Institutional traders often measure execution quality against VWAP:
- Bought below VWAP → good execution (you paid less than the average participant)
- Bought above VWAP → poor execution (you paid more than average)

---
# 5. STOCHASTIC PROCESSES
---

## 5.1 Brownian Motion (Wiener Process)

### The Formula

$$dW_t \sim N(0, dt)$$

or equivalently: $W_{t+dt} - W_t \sim N(0, dt)$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $W_t$ | **Wiener process** (Brownian motion) at time $t$. A continuous random path starting at $W_0 = 0$. |
| $dW_t$ | **Increment** of the Wiener process over an infinitesimal time step $dt$. |
| $N(0, dt)$ | **Normal distribution** with mean 0 and variance $dt$. Standard deviation = $\sqrt{dt}$. |
| $dt$ | **Infinitesimal time step**. In simulation, a small but finite step size. |

### Key Properties

1. **Starts at zero**: $W_0 = 0$
2. **Continuous paths**: No jumps (but infinitely wiggly — never smooth)
3. **Independent increments**: $W_{t_2} - W_{t_1}$ is independent of $W_{t_1} - W_{t_0}$ for $t_0 < t_1 < t_2$
4. **Normal increments**: $W_t - W_s \sim N(0, t - s)$ for $t > s$
5. **Variance grows linearly**: $\text{Var}(W_t) = t$
6. **Not differentiable**: Despite being continuous, the path is too jagged to have a derivative at any point

### Why It Matters in Finance

Brownian motion is the **random engine** driving almost all continuous-time finance models. Stock prices, interest rates, and exchange rates are all modeled using processes built on top of Brownian motion.

## 5.2 Ito Process

### The Formula

$$dX_t = \mu \, dt + \sigma \, dW_t$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $X_t$ | **The stochastic process** at time $t$ — the quantity being modeled. |
| $dX_t$ | **Infinitesimal change** in $X$ over time step $dt$. |
| $\mu$ | **Drift coefficient** — the deterministic trend. Positive $\mu$ means $X$ tends to grow. |
| $dt$ | **Time increment**. |
| $\sigma$ | **Diffusion coefficient** — the scale of randomness. Higher $\sigma$ = more noise. |
| $dW_t$ | **Brownian motion increment** — the source of randomness, $\sim N(0, dt)$. |

### Two Components

$$dX_t = \underbrace{\mu \, dt}_{\text{drift (predictable trend)}} + \underbrace{\sigma \, dW_t}_{\text{diffusion (random noise)}}$$

- **Without noise** ($\sigma = 0$): $dX = \mu \, dt$ → simple linear growth: $X_t = X_0 + \mu t$
- **Without drift** ($\mu = 0$): $dX = \sigma \, dW_t$ → pure random walk, scaled by $\sigma$
- **Both together**: a random walk with a trend

### Euler-Maruyama Simulation

To simulate numerically with discrete steps $\Delta t$:

$$X_{t+\Delta t} = X_t + \mu \cdot \Delta t + \sigma \cdot \sqrt{\Delta t} \cdot Z$$

where $Z \sim N(0, 1)$ is a standard normal random number. This is the **Euler-Maruyama method** — the stochastic equivalent of Euler's method for ODEs.

## 5.3 Ito's Lemma

### The Formula

For $f(t, X_t)$ where $dX_t = \mu \, dt + \sigma \, dW_t$:

$$df = \left(\frac{\partial f}{\partial t} + \mu \frac{\partial f}{\partial x} + \frac{\sigma^2}{2} \frac{\partial^2 f}{\partial x^2}\right) dt + \sigma \frac{\partial f}{\partial x} \, dW_t$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $f(t, X_t)$ | A **smooth function** of time and the stochastic process $X_t$. |
| $\partial f / \partial t$ | How $f$ changes with time (holding $X$ fixed). |
| $\partial f / \partial x$ | How $f$ changes with $X$ (first derivative — the "slope"). |
| $\partial^2 f / \partial x^2$ | How the slope changes with $X$ (second derivative — the "curvature"). |
| $\frac{\sigma^2}{2} \frac{\partial^2 f}{\partial x^2}$ | **The Ito correction term** — THIS is what makes stochastic calculus different from regular calculus. |

### Why Is There an Extra Term?

In regular calculus, the chain rule for $f(X(t))$ is:

$$df = f'(X) \cdot dX$$

In stochastic calculus, because $dW_t$ is "rough" (its square is not negligible — $(dW_t)^2 = dt$), you get an extra term from the Taylor expansion:

$$df \approx f'(X) \cdot dX + \frac{1}{2} f''(X) \cdot (dX)^2$$

Since $(dX)^2 = \sigma^2 (dW)^2 = \sigma^2 \, dt$, this extra term survives and gives the $\frac{\sigma^2}{2} f''$ correction.

### Demonstration with $f(x) = x^2$

If $dX = \mu \, dt + \sigma \, dW$ and $f(X) = X^2$:

- $f'(X) = 2X$
- $f''(X) = 2$

$$d(X^2) = (2X\mu + \sigma^2) \, dt + 2X\sigma \, dW$$

Notice the $\sigma^2$ term — this is purely from the Ito correction. In regular calculus, $d(X^2) = 2X \, dX$ with no extra $\sigma^2$ term.

### Why It Matters

Ito's Lemma is used to derive the Black-Scholes equation, price derivatives, and transform between different stochastic models. It is **the** fundamental tool of mathematical finance.

## 5.4 Mean-Reverting Process (Ornstein-Uhlenbeck)

### The Formula

$$dX_t = \theta(\mu - X_t) \, dt + \sigma \, dW_t$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Effect |
|--------|---------|--------|
| $X_t$ | **Current value** of the process. | |
| $\theta$ (theta) | **Speed of mean reversion** — how fast the process pulls back toward $\mu$. | Higher $\theta$ = faster reversion |
| $\mu$ | **Long-term mean** — the equilibrium level the process oscillates around. | The "magnet" |
| $(\mu - X_t)$ | **Mean reversion force** — the gap between where you are and where you should be. | Positive when $X < \mu$, negative when $X > \mu$ |
| $\theta(\mu - X_t)$ | **Pull strength** — force × speed. Stronger when further from mean. | Like a spring |
| $\sigma$ | **Volatility** — the scale of random perturbations. | |
| $dW_t$ | **Random shock**. | |

### The Spring Analogy

Think of a ball attached to a spring centered at $\mu$:

- When $X > \mu$: the spring pulls DOWN → $(\mu - X) < 0$ → drift is negative
- When $X < \mu$: the spring pulls UP → $(\mu - X) > 0$ → drift is positive
- At $X = \mu$: no pull → drift is zero (but noise still pushes it around)

$\theta$ is the spring constant — higher = stiffer spring = faster return to $\mu$.

### Key Statistical Properties

| Property | Value |
|----------|-------|
| Long-run mean | $E[X_t] \to \mu$ as $t \to \infty$ |
| Long-run variance | $\text{Var}(X_t) \to \frac{\sigma^2}{2\theta}$ |
| Half-life | $t_{1/2} = \frac{\ln 2}{\theta}$ — time for a deviation to decay by half |

### Financial Applications

- **Interest rates** (Vasicek model): rates fluctuate but tend toward a central bank target
- **Volatility** (mean-reverting vol models): vol spikes eventually calm down
- **Pairs trading**: spread between two correlated stocks reverts to historical average

## 5.5 Jump-Diffusion Process

### The Formula

$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t + J \cdot S_t \, dN_t$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $S_t$ | **Asset price** at time $t$. |
| $\mu S_t \, dt$ | **Drift term** — the expected growth rate of the asset, proportional to its current price. |
| $\sigma S_t \, dW_t$ | **Diffusion term** — continuous random fluctuations (geometric Brownian motion). |
| $J$ | **Jump size** — the random magnitude of a jump. Can be fixed or drawn from a distribution (e.g. normal). |
| $dN_t$ | **Poisson process increment** — equals 1 when a jump occurs, 0 otherwise. |
| $\lambda$ (lambda, in $dN_t$) | **Jump intensity** — average number of jumps per unit time. $P(dN_t = 1) \approx \lambda \cdot dt$ for small $dt$. |

### Three Components

$$dS_t = \underbrace{\mu S_t \, dt}_{\text{steady growth}} + \underbrace{\sigma S_t \, dW_t}_{\text{smooth randomness}} + \underbrace{J \cdot S_t \, dN_t}_{\text{sudden jumps}}$$

### Why Add Jumps?

Pure Brownian motion produces **continuous** paths — but real markets have:
- Flash crashes
- Earnings surprises
- Geopolitical shocks

These events cause **discontinuous jumps** that a smooth model can't capture. Jump-diffusion models (like Merton's 1976 model) add realistic sudden moves while keeping the tractability of diffusion models.

### The Poisson Process $dN_t$

- Jumps arrive randomly with average rate $\lambda$ per year
- Each jump is independent
- In a small time step $dt$: probability of exactly 1 jump ≈ $\lambda \cdot dt$, probability of 0 jumps ≈ $1 - \lambda \cdot dt$
- If $\lambda = 3$: expect about 3 jumps per year

## 5.6 Quadratic Variation

### The Formula

$$[X]_T = \lim_{n \to \infty} \sum_{i=0}^{n-1} (X_{t_{i+1}} - X_{t_i})^2$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $[X]_T$ | **Quadratic variation** of process $X$ over $[0, T]$. Measures total accumulated squared movement. |
| $\sum$ | Sum over all partition points. |
| $X_{t_{i+1}} - X_{t_i}$ | **Increment** of the process between consecutive time points. |
| $(\cdot)^2$ | **Squaring** the increment — captures magnitude regardless of direction. |
| $\lim_{n \to \infty}$ | Take finer and finer partitions (infinitely many small time steps). |

### Key Results

| Process | Quadratic Variation $[X]_T$ |
|---------|-----------------------------|
| Smooth function (e.g. $f(t) = t^2$) | **0** — smooth paths have zero QV |
| Brownian motion $W_t$ | **$T$** — this is the defining property |
| Ito process $dX = \mu \, dt + \sigma \, dW$ | **$\sigma^2 T$** — only the diffusion term contributes |

### Why Is This Important?

1. **$(dW_t)^2 = dt$**: This identity (from QV) is what makes Ito's Lemma have the extra correction term
2. **Realized variance estimation**: Computing QV from high-frequency price data gives an estimate of realized variance
3. **Distinguishing stochastic from smooth**: QV is the mathematical proof that Brownian motion paths are fundamentally different from smooth curves

---
# 6. DEPENDENCE & STATISTICS
---

## 6.1 PCA (Principal Component Analysis)

### The Formula

$$\Sigma = V \Lambda V^T$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Dimensions |
|--------|---------|------------|
| $\Sigma$ (Sigma) | **Covariance matrix** — encodes how every pair of variables co-moves. Element $(i,j)$ is the covariance between variable $i$ and $j$. | $p \times p$ (where $p$ = number of variables) |
| $V$ | **Eigenvector matrix** — columns are the principal component directions. Each column points in the direction of one independent source of variance. | $p \times p$ |
| $\Lambda$ (Lambda) | **Diagonal eigenvalue matrix** — $\Lambda_{ii} = \lambda_i$ is the variance explained by the $i$-th principal component. | $p \times p$ diagonal |
| $V^T$ | **Transpose** of $V$ — since eigenvectors are orthonormal, $V^T = V^{-1}$. | $p \times p$ |

### How PCA Works — Step by Step

1. **Compute the covariance matrix** $\Sigma$ from your data
2. **Find eigenvalues and eigenvectors** of $\Sigma$
3. **Sort by eigenvalue** (largest first) — the first eigenvector is the most important direction
4. **Explained variance ratio**: $\frac{\lambda_i}{\sum \lambda_j}$ tells you what fraction of total variance PC$i$ captures

### Finance Application

With 100 stock returns:
- PC1 often captures 40-60% of variance → "the market factor" (everything goes up or down together)
- PC2 might capture 10-15% → "sector rotation" (tech vs energy, growth vs value)
- PC3-5 capture diminishing amounts
- PCs beyond ~5 are mostly noise

### Intuition

Imagine a cloud of data points in 3D space. PCA finds:
- **PC1**: The longest axis of the cloud (direction of maximum spread)
- **PC2**: The second-longest axis, perpendicular to PC1
- **PC3**: Whatever's left, perpendicular to both

If the cloud is flat like a pancake, PC1 and PC2 capture almost everything, and you can safely ignore PC3 (dimensionality reduction).

## 6.2 Kalman Filter (1D Scalar)

### The Formula

**Predict:**
$$\hat{x}_{t|t-1} = A \cdot \hat{x}_{t-1|t-1}$$
$$P_{t|t-1} = A^2 \cdot P_{t-1|t-1} + Q$$

**Update:**
$$K_t = \frac{P_{t|t-1} \cdot H}{H^2 \cdot P_{t|t-1} + R}$$
$$\hat{x}_{t|t} = \hat{x}_{t|t-1} + K_t \cdot (z_t - H \cdot \hat{x}_{t|t-1})$$
$$P_{t|t} = (1 - K_t \cdot H) \cdot P_{t|t-1}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $\hat{x}_{t|t-1}$ | **Predicted state** — our best guess of the hidden state BEFORE seeing the measurement at time $t$. |
| $\hat{x}_{t|t}$ | **Updated state** — our best guess AFTER incorporating the measurement. |
| $P_{t|t-1}$ | **Predicted uncertainty** — how uncertain we are about our prediction. |
| $P_{t|t}$ | **Updated uncertainty** — reduced after seeing the measurement. |
| $K_t$ | **Kalman gain** — a value between 0 and 1 that controls how much we trust the measurement vs our prediction. |
| $z_t$ | **Measurement** (observation) at time $t$ — the noisy data we actually see. |
| $A$ | **State transition** — how the state evolves (for constant state: $A = 1$). |
| $H$ | **Observation model** — how the state maps to measurements (often $H = 1$: we directly observe the state). |
| $Q$ | **Process noise variance** — uncertainty in the state evolution. Higher = state can change more. |
| $R$ | **Measurement noise variance** — uncertainty in the observations. Higher = noisier sensor. |
| $z_t - H \cdot \hat{x}_{t|t-1}$ | **Innovation (residual)** — the surprise: how different the measurement is from what we predicted. |

### The Kalman Gain — The Heart of the Filter

$$K_t = \frac{\text{prediction uncertainty}}{\text{prediction uncertainty} + \text{measurement uncertainty}}$$

| $K_t$ value | Meaning |
|-------------|----------|
| Close to 1 | Trust the measurement more (low $R$, high $Q$) |
| Close to 0 | Trust the prediction more (high $R$, low $Q$) |

### Intuition

You're trying to guess the temperature, but your thermometer is noisy:
1. **Predict**: Based on yesterday's temperature and your model of how temperature changes
2. **Observe**: Read the noisy thermometer
3. **Update**: Blend your prediction with the observation, weighting by their relative reliability
4. **Repeat**: Each step, your estimate gets better as you accumulate more data

## 6.3 Copula

### The Formula

$$F(x, y) = C(F_X(x), F_Y(y))$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| $F(x, y)$ | **Joint distribution** of $X$ and $Y$ — the probability that $X \leq x$ AND $Y \leq y$ simultaneously. |
| $C(u, v)$ | **Copula function** — takes two uniform [0,1] inputs and returns a probability. Encodes the DEPENDENCE STRUCTURE only. |
| $F_X(x)$ | **Marginal CDF of $X$** — the distribution of $X$ alone, ignoring $Y$. Maps $x$ to $[0, 1]$. |
| $F_Y(y)$ | **Marginal CDF of $Y$** — the distribution of $Y$ alone, ignoring $X$. |

### Sklar's Theorem — The Key Insight

Any joint distribution can be decomposed into:
1. **Marginals** (individual distributions) — the "shape" of each variable separately
2. **Copula** (dependence structure) — how the variables are linked together

This separation is powerful because it lets you mix-and-match:
- Use any marginal for $X$ (normal, t, exponential, whatever fits)
- Use any marginal for $Y$
- Choose a copula that captures the right dependence pattern

### Gaussian Copula

The Gaussian copula uses the multivariate normal distribution for dependence:

1. Generate correlated normal samples $(Z_1, Z_2)$ with correlation $\rho$
2. Apply $\Phi$ (normal CDF) to get uniform samples: $U_1 = \Phi(Z_1)$, $U_2 = \Phi(Z_2)$
3. Transform uniform samples to any desired marginal using inverse CDF

### Tail Dependence — Why Copula Choice Matters

| Copula | Tail Dependence | Use Case |
|--------|----------------|----------|
| Gaussian | None | Variables become independent in extremes |
| Student-t | Symmetric | Both tails show dependence (joint crashes AND joint rallies) |
| Clayton | Lower tail only | Assets crash together but don't necessarily rally together |

The 2008 financial crisis exposed the danger of the Gaussian copula in CDO pricing — it underestimated the probability that many borrowers would default simultaneously.

## 6.4 CPPI (Constant Proportion Portfolio Insurance)

### The Formula

$$\text{Risky Exposure} = m \times (\text{Portfolio Value} - \text{Floor})$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Typical Values |
|--------|---------|----------------|
| Risky Exposure | Dollars invested in the risky asset (e.g. stocks). | Varies dynamically |
| $m$ | **Multiplier** — leverage applied to the cushion. Higher = more aggressive. | 2 to 5 |
| Portfolio Value | **Total current value** of your portfolio (risky + safe assets). | |
| Floor | **Minimum acceptable value** — the level you never want to fall below. | Often 80% of initial value |
| Cushion = Portfolio − Floor | **Buffer** between current value and the floor. | ≥ 0 |
| Safe Allocation = Portfolio − Risky Exposure | Remainder invested in safe assets (bonds, cash). | |

### How CPPI Works — The Feedback Loop

1. **Market goes up** → Portfolio rises → Cushion grows → Risky exposure increases → You participate more in the rally
2. **Market goes down** → Portfolio drops → Cushion shrinks → Risky exposure decreases → You de-risk automatically
3. **Near the floor** → Cushion ≈ 0 → Almost entirely in safe assets → Floor is protected

### Example

- Portfolio = $100K, Floor = $80K, $m = 3$
- Cushion = $20K
- Risky exposure = $3 \times 20K = $60K$ in stocks, $40K$ in bonds
- If portfolio drops to $90K: Cushion = $10K → Risky exposure = $30K (automatically de-risked!)

### Gap Risk

If the market drops so fast that the portfolio crashes through the floor before rebalancing can happen (e.g. overnight crash), CPPI fails to protect. This is called **gap risk** and is the main limitation.

## 6.5 APY (Annual Percentage Yield)

### The Formula

$$\text{APY} = \left(1 + \frac{r}{n}\right)^n - 1$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Example |
|--------|---------|----------|
| APY | **Annual Percentage Yield** — the effective annual return after compounding. | 0.0512 = 5.12% |
| $r$ | **Nominal (stated) annual rate** — what the bank advertises. | 0.05 = 5% |
| $n$ | **Compounding frequency** — how many times per year interest is calculated and added to the balance. | 12 = monthly, 365 = daily |
| $r/n$ | **Rate per compounding period**. | 0.05/12 ≈ 0.00417 per month |
| $(1 + r/n)$ | **Growth factor per period**. | 1.00417 |
| $(\cdot)^n$ | **Compound $n$ times** — the power of compounding! | $(1.00417)^{12}$ |
| $-1$ | Subtract the initial principal to get the net yield. | |

### Compounding Frequency Effect

For a 5% nominal rate:

| Compounding | $n$ | APY |
|-------------|-----|------|
| Annual | 1 | 5.0000% |
| Semi-annual | 2 | 5.0625% |
| Monthly | 12 | 5.1162% |
| Daily | 365 | 5.1267% |
| Continuous | $\to \infty$ | $e^r - 1$ = 5.1271% |

### Why APY Matters

Two banks both advertise "5% interest" but one compounds monthly and the other annually. APY lets you make an apples-to-apples comparison. The monthly-compounding bank actually gives you more (5.116% vs 5.000%).

## 6.6 Capital Efficiency

### The Formula

$$\text{CE} = \frac{\text{Return}}{\text{Capital Used}}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| CE | **Capital Efficiency** — return generated per unit of capital deployed. |
| Return | The profit or yield generated by the activity. |
| Capital Used | The total amount of capital deployed to generate that return. |

### Context in DeFi

In DeFi, capital efficiency is critical because liquidity providers lock up capital:

| Protocol | Capital Locked | Fees Earned | CE |
|----------|---------------|-------------|-----|
| Pool A | $1,000,000 | $50,000 | 5% |
| Pool B (concentrated) | $200,000 | $40,000 | 20% |

Pool B is 4× more capital efficient — it generates nearly as much with 1/5 the capital.

This is why Uniswap V3 introduced **concentrated liquidity** — LPs choose a price range, making their capital work harder within that range.

---
# 7. DeFi / AMM
---

## 7.1 Impermanent Loss

### The Formula

$$\text{IL} = \frac{2\sqrt{P}}{1 + P} - 1$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Example |
|--------|---------|----------|
| IL | **Impermanent Loss** — the percentage loss compared to just holding the tokens. Always ≤ 0 (or exactly 0 when P=1). | -0.0566 = -5.66% |
| $P$ | **Price ratio** = current price / entry price. If the token doubled, $P = 2$. If it halved, $P = 0.5$. | > 0 |
| $\sqrt{P}$ | The AMM rebalances such that token quantities adjust with the square root of the price ratio. | |
| $2\sqrt{P}/(1+P)$ | The ratio of AMM portfolio value to hold portfolio value. Always ≤ 1. | |

### Derivation Intuition

A constant-product AMM (like Uniswap V2) maintains $x \cdot y = k$ where $x$ and $y$ are token quantities. When the price changes:

- **HODL portfolio**: You still have the same tokens, valued at new prices
- **AMM portfolio**: The pool rebalanced — you have more of the cheaper token and less of the expensive one

The AMM portfolio is ALWAYS worth less than or equal to the HODL portfolio (unless price returns to entry).

### IL for Common Price Changes

| Price Change | $P$ | IL |
|-------------|------|------|
| No change | 1.0 | 0% |
| +25% | 1.25 | -0.60% |
| +50% | 1.50 | -2.02% |
| +100% (2×) | 2.00 | -5.72% |
| +200% (3×) | 3.00 | -13.40% |
| +400% (5×) | 5.00 | -25.46% |
| -50% | 0.50 | -5.72% |
| -75% | 0.25 | -20.00% |

### Key Insight

IL is **symmetric with respect to reciprocal price moves**: doubling ($P=2$) and halving ($P=0.5$) give the same IL. This is because the AMM structure is symmetric — it doesn't care which direction the price moves, only *how much* it diverges from entry.

### When Is IL "Permanent"?

It's called "impermanent" because if the price returns to entry ($P=1$), the loss disappears. But if you withdraw while $P \neq 1$, the loss is realized and very much permanent.

## 7.2 Health Factor

### The Formula

$$\text{HF} = \frac{\text{Collateral} \times \text{Liquidation Threshold}}{\text{Debt}}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Example |
|--------|---------|----------|
| HF | **Health Factor** — a unitless safety measure. HF > 1 = safe, HF < 1 = can be liquidated. | 1.6 |
| Collateral | **Dollar value** of assets deposited as collateral. | $10,000 |
| Liquidation Threshold | **Maximum LTV** allowed before liquidation — set by the protocol per asset type. | 0.80 (80%) |
| Collateral × LT | **Effective collateral** — the borrowing power of your collateral. | $8,000 |
| Debt | **Total amount borrowed** (in dollar value). | $5,000 |

### Reading the Health Factor

| HF | Status | Meaning |
|----|--------|----------|
| > 2.0 | Very safe | Large buffer — collateral can drop significantly |
| 1.5 - 2.0 | Moderate | Comfortable but monitor |
| 1.0 - 1.5 | Warning | Getting close to liquidation zone |
| ≤ 1.0 | **Liquidation** | Protocol can seize and sell your collateral |

### How Liquidation Works

When HF drops below 1.0:
1. Anyone can call the liquidation function on your position
2. The liquidator repays part of your debt
3. In return, they receive your collateral at a discount (the liquidation bonus, typically 5-15%)
4. You lose collateral and the discount is the liquidator's profit

## 7.3 Borrow Rate (Kinked Utilization Model)

### The Formula

$$r(u) = \begin{cases} r_{\text{base}} + \text{slope}_1 \cdot \frac{u}{u_{\text{kink}}} & \text{if } u \leq u_{\text{kink}} \\ r_{\text{base}} + \text{slope}_1 + \text{slope}_2 \cdot \frac{u - u_{\text{kink}}}{1 - u_{\text{kink}}} & \text{if } u > u_{\text{kink}} \end{cases}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Typical Value |
|--------|---------|---------------|
| $r(u)$ | **Borrow rate** as a function of utilization. | e.g. 6% annual |
| $u$ | **Utilization rate** = Total Borrowed / Total Deposited. How much of the pool is lent out. | 0 to 1 |
| $r_{\text{base}}$ | **Base rate** — the minimum borrow cost even at 0 utilization. | 2% |
| $\text{slope}_1$ | **Gentle slope** — how fast rates rise below the kink. | 4% |
| $\text{slope}_2$ | **Steep slope** — how fast rates spike above the kink. Much larger than slope1. | 75% |
| $u_{\text{kink}}$ | **Optimal utilization** — the threshold where the rate model shifts from gentle to aggressive. | 80% |

### Why the Kink?

The kink creates a **two-regime model**:

- **Below kink** ($u < 80\%$): Rates rise gently — borrowing is encouraged, system is healthy
- **Above kink** ($u > 80\%$): Rates spike dramatically — borrowers face painful costs, incentivizing repayment

The purpose: **protect depositors**. If utilization hits 100%, depositors cannot withdraw their funds. The steep slope above the kink makes borrowing extremely expensive, pushing utilization back down.

### Real Example (Aave-style)

At 90% utilization: rate ≈ 43.5% APR → borrowers rush to repay  
At 80% utilization: rate ≈ 6% APR → comfortable for everyone  
At 50% utilization: rate ≈ 4.5% APR → cheap to borrow

## 7.4 TVL (Total Value Locked)

### The Formula

$$\text{TVL} = \sum_{i=1}^{n} q_i \times p_i$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| TVL | **Total Value Locked** — the aggregate dollar value of all assets deposited in the protocol. |
| $q_i$ | **Quantity** of asset $i$ held in the protocol's smart contracts. |
| $p_i$ | **Current price** of asset $i$ in USD (or another reference currency). |
| $q_i \times p_i$ | **Dollar value** of asset $i$'s holdings. |
| $\sum$ | **Sum** over all $n$ different assets in the protocol. |

### What TVL Tells You

TVL is the DeFi equivalent of "assets under management" (AUM) in traditional finance.

| TVL | Interpretation |
|-----|----------------|
| Growing | Users are depositing → trust / adoption increasing |
| Shrinking | Users are withdrawing → possible concerns, better opportunities elsewhere |
| High absolute | Large, established protocol |
| Low absolute | New, niche, or risky protocol |

### Caveats

1. **Double-counting**: If Protocol A deposits into Protocol B, the TVL is counted in both. Industry-wide TVL can be inflated.
2. **Token price dependence**: If the locked token's price drops 50%, TVL drops 50% even with no withdrawals.
3. **TVL ≠ revenue**: A protocol can have billions in TVL but generate minimal fees.

---
# 8. TRADING / EXECUTION
---

## 8.1 Slippage

### The Formula

$$\text{Slippage} = P_{\text{exec}} - P_{\text{expected}}$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning |
|--------|---------|
| Slippage | **The cost of imperfect execution** — the gap between the price you wanted and the price you got. | 
| $P_{\text{exec}}$ | **Execution price** — the actual fill price of your order. |
| $P_{\text{expected}}$ | **Expected price** — the price you saw when you decided to trade (e.g. the mid-price or the quoted price). |

### Signed Convention

For a **buy** order:
- Slippage > 0 → You paid MORE than expected → **bad** (cost you money)
- Slippage < 0 → You paid LESS than expected → **good** (price improvement)

For a **sell** order:
- Slippage > 0 → You received MORE than expected → **good**
- Slippage < 0 → You received LESS than expected → **bad**

### Expressing in Basis Points

$$\text{Slippage (bps)} = \frac{P_{\text{exec}} - P_{\text{expected}}}{P_{\text{expected}}} \times 10{,}000$$

One basis point = 0.01%. A slippage of 5 bps on a $1M trade costs $500.

### Causes of Slippage

1. **Market movement**: Price changes between decision and execution
2. **Order book depth**: Not enough liquidity at the best price → your order walks up the book
3. **Latency**: Delay in order transmission
4. **Large order size**: Your order is big enough to move the price (see: Price Impact)

## 8.2 Price Impact

### The Formula

$$\Delta P = \lambda \cdot Q$$

### Symbol-by-Symbol Breakdown

| Symbol | Meaning | Units |
|--------|---------|-------|
| $\Delta P$ | **Price impact** — how much the market price moves because of your order. | Dollars per share |
| $\lambda$ (lambda) | **Market impact coefficient** — a constant that captures how sensitive the market is to order flow. Higher = more illiquid market. | Dollars per share per unit of order size |
| $Q$ | **Order size** — the number of shares (or contracts, or tokens) you are buying or selling. | Shares |

### Linear vs Real-World Impact

The formula $\Delta P = \lambda Q$ is the simplest (linear) model. In practice:

- **Square-root model** (more realistic): $\Delta P = \lambda \cdot \sigma \cdot \sqrt{Q/V}$ where $V$ = daily volume and $\sigma$ = daily volatility
- Impact is usually **concave** — the first 1000 shares have more impact per share than the last 1000 of a 10,000-share order

### Temporary vs Permanent Impact

| Type | Meaning | Duration |
|------|---------|----------|
| Temporary | Price spike caused by your order consuming liquidity | Fades in minutes to hours as market makers refill the book |
| Permanent | Information content of your order shifts the fair price | Persists indefinitely — the market learned something from your trade |

### Why It Matters

For a hedge fund trading $100M:
- 10 bps of price impact = $100,000 cost per trade
- Trading 250 days/year = $25M/year in impact costs
- This can be the difference between a profitable and unprofitable strategy

### Minimizing Price Impact

1. **Split the order**: Use TWAP/VWAP algorithms to spread execution over time
2. **Use dark pools**: Trade anonymously to avoid signaling your intentions
3. **Trade liquid names**: Impact is inversely related to average daily volume
4. **Be patient**: Faster execution = more impact

---
# Summary: Formula Quick Reference
---

| # | Section | Formula | Key Idea |
|---|---------|---------|----------|
| 1.1 | Volatility | $\sigma = \sqrt{\frac{1}{n-1}\sum(r_i - \bar{r})^2}$ | How spread out returns are |
| 1.2 | Volatility | Solve $C_{mkt} = C_{BS}(\sigma_{impl})$ | Market's forecast of future vol |
| 1.3 | Volatility | $\sigma^2_{fwd} = \frac{\sigma_2^2 T_2 - \sigma_1^2 T_1}{T_2 - T_1}$ | Vol expected in a future window |
| 1.4 | Volatility | $R = \prod(1+r_i) - 1$ | Total compounded return |
| 2.1 | Risk | $VaR = \mu - z_\alpha \sigma$ | Worst expected loss at confidence $\alpha$ |
| 2.2 | Risk | $Sharpe = \frac{E[R]-R_f}{\sigma}$ | Return per unit of total risk |
| 2.3 | Risk | $Sortino = \frac{E[R]-R_f}{\sigma_{down}}$ | Return per unit of downside risk |
| 2.4 | Risk | $RAROC = \frac{Return}{EconCapital}$ | Return per unit of risk capital |
| 3.1 | Options | $C = SN(d_1) - Ke^{-rT}N(d_2)$ | Fair price of a European call |
| 3.2 | Options | $\Delta = N(d_1)$ | Option sensitivity to stock price |
| 3.3 | Options | $\Gamma = N'(d_1)/(S\sigma\sqrt{T})$ | Rate of change of delta |
| 3.4 | Options | $\nu = SN'(d_1)\sqrt{T}$ | Option sensitivity to volatility |
| 4.1 | Credit | $LTV = Loan / Collateral$ | Borrowing as fraction of collateral |
| 4.2 | Credit | $EL = PD \times LGD \times EAD$ | Average expected loss on a loan |
| 4.3 | Execution | $TWAP = \frac{1}{T}\sum P(t)$ | Equal-weighted average price |
| 4.4 | Execution | $VWAP = \frac{\sum P_i V_i}{\sum V_i}$ | Volume-weighted average price |
| 5.1 | Stochastic | $dW \sim N(0,dt)$ | Foundation of random models |
| 5.2 | Stochastic | $dX = \mu dt + \sigma dW$ | Random walk with trend |
| 5.3 | Stochastic | Ito's Lemma (chain rule + correction) | Calculus for random processes |
| 5.4 | Stochastic | $dX = \theta(\mu-X)dt + \sigma dW$ | Process pulled toward a mean |
| 5.5 | Stochastic | $dS = \mu S dt + \sigma S dW + JS dN$ | Diffusion plus sudden jumps |
| 5.6 | Stochastic | $[X]_T = \sum(\Delta X)^2$ | Accumulated squared movement |
| 6.1 | Statistics | $\Sigma = V\Lambda V^T$ | Find independent variance sources |
| 6.2 | Statistics | $\hat{x} = \hat{x}_{pred} + K(z - H\hat{x}_{pred})$ | Optimal estimate from noisy data |
| 6.3 | Statistics | $F(x,y) = C(F_X(x), F_Y(y))$ | Separate marginals from dependence |
| 6.4 | Statistics | $Exposure = m(Portfolio - Floor)$ | Dynamic downside protection |
| 6.5 | Statistics | $APY = (1+r/n)^n - 1$ | True annual yield with compounding |
| 6.6 | Statistics | $CE = Return / Capital$ | Return per dollar deployed |
| 7.1 | DeFi | $IL = 2\sqrt{P}/(1+P) - 1$ | LP loss vs holding tokens |
| 7.2 | DeFi | $HF = (Coll \times LT) / Debt$ | Distance from liquidation |
| 7.3 | DeFi | Kinked rate model | Rates spike above optimal utilization |
| 7.4 | DeFi | $TVL = \sum q_i p_i$ | Total capital in a protocol |
| 8.1 | Trading | $Slip = P_{exec} - P_{exp}$ | Cost of imperfect execution |
| 8.2 | Trading | $\Delta P = \lambda Q$ | How your order moves the market |